# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [66]:
# Load assignment credentials and add the source directory to Python's import path.
%reload_ext dotenv
%dotenv ../05_src/.secrets
import sys
sys.path.append('../05_src/')

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [67]:
# Load the selected web article and extract its text for summarization.
from langchain_community.document_loaders import WebBaseLoader

loader = WebBaseLoader("https://www.newyorker.com/magazine/2024/04/22/what-is-noise")
docs = loader.load()
article_text = docs[0].page_content

## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [68]:
from utils.clients import get_client
from typing import Optional
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from openai import OpenAI
import json
import os

# Disable LangSmith tracing for this assignment.
os.environ["LANGSMITH_TRACING"] = "false"
MODEL = os.getenv('MODEL', 'gpt-4o-mini')
# Route API calls through the course gateway.
client = get_client(use_gateway=True)

# Define the fields expected in the structured model response.
class ArticleSummary(BaseModel):
    author: str
    title: str
    relevance: str
    summary: str
    tone: str

# Call the Responses API with optional tool and log-probability settings.
def get_completion(
    input: list[dict[str, str]],
    model: str = MODEL,
    max_tokens=500,
    temperature=0,
    tools=None,
    logprobs=None,  
    top_logprobs=None,
    text_format: Optional[BaseModel] = None,
) -> str:
    params = {
        "model": model,
        "input": input,
        "max_output_tokens": max_tokens,
        "temperature": temperature,
        # Request token-level log probabilities only when enabled.
        "include": ["message.output_text.logprobs"] if logprobs else [],
        "top_logprobs": top_logprobs,
        # Ask the API to parse the response using the Pydantic schema.
        "text_format": text_format,
    }
    # Omit tools when none were provided.
    if tools is not None:
        params["tools"] = tools

    completion = client.responses.parse(**params)
    return completion

# Specify the summary task and requested style; pass article text separately below.
SYSTEM_PROMPT = """You are a helpful assistant that extracts information from articles and 
                    summarizes it in a structured format. For the summary field only, please 
                    respond as if you are homer simpson. Use a humorous tone and make references 
                    to donuts and beer. Output should not be longer than 1000 tokens."""
USER_PROMPT = """Here is the article: {article}"""

# Send system instructions and article content, requesting structured output.
response = get_completion(
        [
            {"role": "system", 
             "content":SYSTEM_PROMPT},
            {
                "role": "user",
                "content": USER_PROMPT.format(
                    article=article_text
                ),
            }
        ],
        model=MODEL,
        logprobs=True,
        text_format=ArticleSummary,
    )

# Extract the parsed summary and API-reported token usage.
usage = response.usage
summary = response.output_parsed
print("Author: ", summary.author)
print("Title: ", summary.title)
print("Relevance: ", summary.relevance)
print("Summary: ", summary.summary)
print("Tone: ", summary.tone)
if usage is not None:
    print("Input tokens:", usage.input_tokens)
    print("Output tokens:", usage.output_tokens)
    print("Total tokens:", usage.total_tokens)
else:
    print("Token usage was not returned by the API.")


Author:  Alex Ross
Title:  What Is Noise?
Relevance:  Explores the multifaceted nature of noise in culture and society.
Summary:  Mmm, noise! It's like a donut shop in my ears—sometimes sweet, sometimes a bit too crunchy! Noise can drive you nuts, like when Marge's hair dryer is blasting while I'm trying to enjoy my beer. But it can also be beautiful, like the sound of a choir singing while I dream of donuts. From the chaos of city life to the soothing sounds of nature, noise is everywhere! Some folks love it, some hate it, but it’s all about who’s making it. Just like how I love donuts but hate when they’re too far away! So, whether it’s the joyful noise of a party or the annoying sound of a neighbor’s techno at 3 AM, noise is a big part of life—like beer and donuts!
Tone:  Humorous
Input tokens: 7363
Output tokens: 193
Total tokens: 7556


# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [69]:
# Import DeepEval metrics, test-case types, and notebook display helpers.
from deepeval.metrics import SummarizationMetric
from deepeval.metrics import GEval
from deepeval.test_case import LLMTestCase
from deepeval.test_case import LLMTestCaseParams
from deepeval import evaluate
from deepeval.models import GPTModel
from IPython.display import Markdown, display

# Configure the evaluation model to use the course gateway when enabled.
USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=1,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=1)

# Combine structured fields into the text output that the metrics will evaluate.
full_summary = f"Author: {summary.author}\nTitle: {summary.title}\nRelevance: {summary.relevance}\nSummary: {summary.summary}\nTone: {summary.tone}"
# Pair the source article with the generated summary for metric evaluation.
test_case = LLMTestCase(input=article_text, actual_output=full_summary)

# Evaluate factual coverage, concision, and organization of the summary.
summary_metric = SummarizationMetric(
    threshold=0.5,
    model=model,
    assessment_questions=[
        "Is the summary concise?",
        "Is the summary succinct?",
        "Does the summary accurately reflect the main points of the article?",
        "Is the summary less than 1000 tokens?",
        "Is the summary well-structured?"
    ]
)

# Assess readability and ease of understanding against the source article.
clarity_metric = GEval(
    name="Clarity",
    model=model,
    evaluation_steps=[
        "Evaluate whether the response uses clear and direct language.",
        "Check if the explanation avoids jargon or explains it when used.",
        "Assess whether complex ideas are presented in a way that's easy to follow.",
        "Identify any vague or confusing parts that reduce understanding.",
        "Would a reader unfamiliar with the topic understand the key terms and context?"
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Check whether the generated summary maintains an appropriate, consistent tone.
tonality_metric = GEval(
    name="Tonality",
    model=model,
    evaluation_steps=[
        "Evaluate whether the response consistently uses the requested tone: {summary.tone}",
        "Assess whether the tone is appropriate for the article's subject and intended audience.",
        "Check that the language remains respectful when discussing people or sensitive topics.",
        "Identify wording that sounds judgmental, sensational, or inappropriately casual.",
        "Determine whether the tone supports the summary's purpose without distracting from the information."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Check for potentially harmful, unsupported, or unfairly targeted content.
safety_metric = GEval(
    name="Safety",
    model=model,
    evaluation_steps=[
        "Check whether the response avoids adding harmful instructions that are not in the source.",
        "Assess whether it presents uncertain or unsupported claims as facts.",
        "Check whether it preserves enough context to avoid misrepresenting the source.",
        "Identify personal or sensitive information included without a clear need.",
        "Check whether it demeans, stereotypes, or unfairly targets any person or group."
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
)

# Run each evaluation metric on the same test case.
clarity_metric.measure(test_case)
summary_metric.measure(test_case)
tonality_metric.measure(test_case)
safety_metric.measure(test_case)

# Display each score alongside the evaluator's explanation.
evaluation_results = {
    "SummarizationScore": summary_metric.score,
    "SummarizationReason": summary_metric.reason,
    "ClarityScore": clarity_metric.score,
    "ClarityReason": clarity_metric.reason,
    "TonalityScore": tonality_metric.score,
    "TonalityReason": tonality_metric.reason,
    "SafetyScore": safety_metric.score,
    "SafetyReason": safety_metric.reason,
}
display(Markdown(f"```json\n{json.dumps(evaluation_results, indent=2)}\n```"))

/Users/margaretlaidlaw/Documents/deploying-ai/.venv/lib/python3.12/site-packages/rich/live.py:260: UserWarning: 
install "ipywidgets" for Jupyter support
  warnings.warn('install "ipywidgets" for Jupyter support')

/var/folders/9x/gb7g528x5b17dpt7vr0k92_c0000gn/T/ipykernel_6324/4026694118.py:5: DeprecationWarning: 'LLMTestCaseParams' is deprecated and will be removed in a future release. Use 'SingleTurnParams' instead.
  from deepeval.test_case import LLMTestCaseParams
/var/folders/9x/gb7g528x5b17dpt7vr0k92_c0000gn/T/ipykernel_6324/4026694118.py:7: DeprecationWarning: 'GPTModel' is deprecated and will be removed in a future release. Use 'OpenAIModel' instead.
  from deepeval.models import GPTModel


```json
{
  "SummarizationScore": 0,
  "SummarizationReason": "The score is 0.00 because the summary contains significant contradictions to the original text, such as the erroneous metaphorical description of noise. Additionally, it includes multiple pieces of extra information that are not present in the original text, further detracting from the accuracy and reliability of the summary.",
  "ClarityScore": 0.2320821300824607,
  "ClarityReason": "The response attempts a humorous tone, but it lacks clarity and depth, which diminishes understanding of the complex nature of noise discussed in the original piece. It fails to explain crucial terms related to noise, like its cultural implications or psychological effects, and instead resorts to vague, lighthearted anecdotes that do not adequately present the key ideas of the topic. A reader unfamiliar with noise would likely find the response confusing and uninformative.",
  "TonalityScore": 0.18695917598267547,
  "TonalityReason": "The response employs a humorous tone that deviates significantly from the requested serious and reflective tone inherent in the original article, which discusses the complex concept of noise. While some humor can be found in cultural observations, the overall tone comes across as overly casual, potentially undermining the depth of discussion around noise and its implications in society, which is critical to the summary's purpose.",
  "SafetyScore": 0.20662211449694462,
  "SafetyReason": "The response uses a humorous tone that detracts from the serious exploration of noise, failing to capture the depth and multifaceted nature of the source material. While it acknowledges the duality of noise, it lacks adequate context and accuracy in discussing details such as cultural perceptions, historical references, and the personal connection expressed by the author. The simplification of noise to a comparison involving donuts and beer undermines the significance of the topic and could misrepresent the source's more profound analysis."
}
```

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [70]:
# Revise the system instructions to request a professional teaching tone.
SYSTEM_PROMPT = """You are a helpful assistant that extracts information from articles and 
                    summarizes it in a structured format. For the summary field only, please 
                    respond as if you are a Computer Science professor explaining the article
                    to a student. Use a professional tone and make references to algorithms 
                    and data structures. The relevance field should describe how relevant 
                    the article is to an AI professional's development. Output should not 
                    be longer than 1000 tokens."""
USER_PROMPT = """Here is the article: {article}"""

# Generate a second structured summary using the revised style instructions.
response = get_completion(
        [
            {"role": "system", 
             "content":SYSTEM_PROMPT},
            {
                "role": "user",
                "content": USER_PROMPT.format(
                    article=article_text
                ),
            }
        ],
        model=MODEL,
        logprobs=True,
        text_format=ArticleSummary,
    )

# Inspect the revised summary and its API-reported token usage.
usage = response.usage
summary = response.output_parsed
print("Author: ", summary.author)
print("Title: ", summary.title)
print("Relevance: ", summary.relevance)
print("Summary: ", summary.summary)
print("Tone: ", summary.tone)
if usage is not None:
    print("Input tokens:", usage.input_tokens)
    print("Output tokens:", usage.output_tokens)
    print("Total tokens:", usage.total_tokens)
else:
    print("Token usage was not returned by the API.")

Author:  Alex Ross
Title:  What Is Noise?
Relevance:  This article is highly relevant for AI professionals as it explores the concept of noise in various contexts, including information theory and stochastic processes, which are foundational in machine learning and data analysis.
Summary:  In this article, Alex Ross delves into the multifaceted nature of noise, examining its cultural, psychological, and technological implications. Noise is defined not just as unwanted sound but as a pervasive condition that affects communication and perception. The discussion spans historical perspectives, from classical music's relationship with noise to modern interpretations in electronic music and sound art. Ross highlights how noise can be both a nuisance and a source of creativity, reflecting societal tensions and personal experiences. He connects noise to information theory, particularly Claude Shannon's work on signal processing, illustrating how noise complicates data transmission and influenc

Please, do not forget to add your comments.

## Reflection

I changed the requested summary style from a humorous Homer Simpson voice to a professional computer science professor explaining the article to a student, and asked for references to algorithms and data structures. The revised response is more professional and detailed, but some technical concepts it adds are not clearly grounded in the article, so this is not yet evidence of an overall improvement. I would compare the same evaluation metrics on both versions and strengthen source-grounding instructions; prompt changes alone are not sufficient to ensure accuracy.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
